Analyzed padding-and-strides.ipynb:L1-298

本章节的核心内容是卷积神经网络中调节特征图空间维度的两个关键超参数：**填充（Padding）** 与 **步幅（Stride）**。以下是该 Notebook 的系统总结：

---

### 一、核心动机

1. **为什么需要填充（Padding）？**
   * 普通卷积核大小通常大于 1（如 3×3、5×5），经过连续多层卷积后，图像边缘的像素参与计算的次数远少于中心像素，导致特征图尺寸逐层缩小，丢失大量边界信息。
   * 填充通过在输入边缘补零来保护边缘信息，常用于**维持输入与输出的空间尺寸一致**。

2. **为什么需要步幅（Stride）？**
   * 默认情况下卷积窗口每次只移动 1 个元素。如果原始图像分辨率极高或信息冗余，需要大幅下采样以降低空间尺寸与后续计算量。
   * 步幅通过让滑动窗口跳过部分元素，实现**成倍压缩特征图尺寸**。

---

### 二、填充（Padding）

#### 1. 尺寸计算公式
假设输入尺寸为 `n_h × n_w`，卷积核尺寸为 `k_h × k_w`，在高度方向填充 `p_h` 行（通常上下各加约一半），在宽度方向填充 `p_w` 列（通常左右各加约一半）：

```text
输出高度 H_out = n_h - k_h + p_h + 1
输出宽度 W_out = n_w - k_w + p_w + 1
```

#### 2. 常用设计习惯
* **保持输入输出尺寸相同**：令 `p_h = k_h - 1`，`p_w = k_w - 1` 即可保证输出高宽与输入一致。
* **优先选择奇数卷积核（1, 3, 5, 7 等）**：
  * 对称性：当卷积核为奇数时，`k - 1` 为偶数，可以在上下、左右两侧对称填充相同行/列数（两边各填 `(k - 1) / 2`）。
  * 中心对齐：输出的每一个点 `Y[i, j]` 恰好对应以输入 `X[i, j]` 为中心与卷积核的互相关计算。

#### 3. PyTorch 中的参数规则
* 在 PyTorch 的 `nn.Conv2d(..., padding=1)` 中，参数 `padding=1` 表示**单侧填充 1 行/列**，因此输入高和宽两端各加 1，总填充数 `p_h = 2`、`p_w = 2`。

---

### 三、步幅（Stride）

#### 1. 完整输出尺寸公式
当高度方向步幅为 `s_h`、宽度方向步幅为 `s_w` 时，输出形状为：

```text
输出高度 H_out = ⌊(n_h - k_h + p_h + s_h) / s_h⌋
输出宽度 W_out = ⌊(n_w - k_w + p_w + s_w) / s_w⌋
```

* **特例 1（输入输出保尺寸的前提下步幅采样）**：若取 `p_h = k_h - 1`，公式简化为 `⌊(n_h + s_h - 1) / s_h⌋`。
* **特例 2（整除情况）**：若输入尺寸可被步幅整除，输出尺寸直接等于 `(n_h / s_h) × (n_w / s_w)`。

---

### 四、代码示例要点

Notebook 中通过定义辅助函数 `comp_conv2d` 验证了不同组合下的维度变化（输入张量 `X` 均为 8×8）：

| 场景 | 卷积核 `kernel_size` | 填充 `padding` (单侧) | 步幅 `stride` | 输出形状 | 说明 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **基础对称填充** | 3×3 | 1（两边共+2） | 1 | `[8, 8]` | 保持与输入尺寸一致 |
| **非对称核与填充** | 5×3 | (2, 1)（高+4, 宽+2） | 1 | `[8, 8]` | 核高5配填充4，核宽3配填充2，保持尺寸 |
| **步幅下采样** | 3×3 | 1（两边共+2） | 2 | `[4, 4]` | 高宽尺寸均被减半 |
| **非对称步幅与填充** | 3×5 | (0, 1)（高+0, 宽+2） | (3, 4) | `[2, 2]` | 满足复杂向下取整计算公式 |

> **第四个示例的验算**：
> * 高度：`⌊(8 - 3 + 0 + 3) / 3⌋ = ⌊8 / 3⌋ = 2`
> * 宽度：`⌊(8 - 5 + 2 + 4) / 4⌋ = ⌊9 / 4⌋ = 2`

---

### 五、总结

* **填充（Padding）**：用于**防止尺寸过快缩减**并保留图像边缘信息，常用来让特征图在卷积前后保持相同空间分辨率。
* **步幅（Stride）**：用于**主动降低特征图分辨率**，大幅缩减后续计算量与参数负担。
* 工程实践中通常采用**对称的填充与步幅**（即 `p_h = p_w` 且 `s_h = s_w`），非对称形式主要用于特定结构（如一维卷积、时序特征或特定感受野设计）。

In [1]:
import torch
from torch import nn

In [2]:

def comp_conv2d(conv2d, X):
    X = X.reshape((1,1)+X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

conv2d = nn.Conv2d(1,1, kernel_size=3, padding=1)
X = torch.rand(size=(8,8))
comp_conv2d(conv2d, X).shape


torch.Size([8, 8])

In [4]:
conv2d = nn.Conv2d(1,1, kernel_size=(5,3), padding=(2,1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

In [5]:
conv2d = nn.Conv2d(1,1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

In [6]:
conv2d = nn.Conv2d(1,1, kernel_size=(3,5), padding=(0,1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

以下是本节课后 4 道习题的详细解答与分析：

---

### 习题 1：对于本节中的最后一个示例，计算其输出形状，以查看它是否与实验结果一致。

#### 【解答】
在 notebook 的最后一个示例中，各项参数如下：
* **输入形状**：`n_h = 8, n_w = 8`
* **卷积核形状**：`k_h = 3, k_w = 5`
* **填充**：`padding = (0, 1)`。注意 PyTorch 的 `padding` 指定的是单侧填充数量，因此：
  * 高度总填充：`p_h = 0 × 2 = 0`
  * 宽度总填充：`p_w = 1 × 2 = 2`
* **步幅**：`stride = (3, 4)`，即 `s_h = 3, s_w = 4`

通用输出形状公式：
```text
H_out = ⌊(n_h - k_h + p_h + s_h) / s_h⌋
W_out = ⌊(n_w - k_w + p_w + s_w) / s_w⌋
```

代入具体数值：
```text
输出高度 H_out = ⌊(8 - 3 + 0 + 3) / 3⌋
              = ⌊8 / 3⌋ 
              = ⌊2.666...⌋ = 2

输出宽度 W_out = ⌊(8 - 5 + 2 + 4) / 4⌋
              = ⌊9 / 4⌋ 
              = ⌊2.25⌋ = 2
```

**结论**：理论计算得到的输出形状为 `2 × 2`，与实验中 `torch.Size([2, 2])` 完全一致。

---

### 习题 2：在本节中的实验中，试一试其他填充和步幅组合。

#### 【解答】
我们可以设计三组具有代表性的组合，分别观察「放大特征图」、「奇数输入下采样」和「经典下采样」的计算与代码验证：

```python
import torch
from torch import nn

def comp_conv2d(conv2d, X):
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:])

# 组合 1：大填充 (padding > (k-1)/2)，输出尺寸反超输入尺寸
# 输入 8×8, 核 3×3, padding=2 (单侧2, 总+4), stride=1
# 理论高度: 8 - 3 + 4 + 1 = 10 -> [10, 10]
conv1 = nn.Conv2d(1, 1, kernel_size=3, padding=2, stride=1)
print("组合 1 输出:", comp_conv2d(conv1, torch.rand(8, 8)).shape)  # torch.Size([10, 10])

# 组合 2：输入尺寸为奇数，配合步幅 2
# 输入 7×7, 核 3×3, padding=1 (总+2), stride=2
# 理论高度: ⌊(7 - 3 + 2 + 2) / 2⌋ = ⌊8 / 2⌋ = 4 -> [4, 4]
conv2 = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)
print("组合 2 输出:", comp_conv2d(conv2, torch.rand(7, 7)).shape)  # torch.Size([4, 4])

# 组合 3：无填充非对称核下采样（常用于降低某一特定维度）
# 输入 9×9, 核 (3, 5), padding=(1, 0) (高总+2, 宽总+0), stride=(2, 1)
# 理论高度: ⌊(9 - 3 + 2 + 2) / 2⌋ = ⌊10 / 2⌋ = 5
# 理论宽度: ⌊(9 - 5 + 0 + 1) / 1⌋ = 5 -> [5, 5]
conv3 = nn.Conv2d(1, 1, kernel_size=(3, 5), padding=(1, 0), stride=(2, 1))
print("组合 3 输出:", comp_conv2d(conv3, torch.rand(9, 9)).shape)  # torch.Size([5, 5])
```

---

### 习题 3：对于音频信号，步幅 2 说明什么？

#### 【解答】
音频信号本质上是一维连续时序信号（离散化后为 1D 序列，形如 `[采样点数]`）或二维时频图（如梅尔频谱 Spectrogram，形如 `[频段, 时间帧]`）。

当步幅设为 2 时，具体含义如下：
1. **时间维度分辨率减半（降采样 / Downsampling）**：
   * 在一维卷积中，步幅 2 意味着卷积核每次跳过 1 个采样点，仅对每隔一个位置进行特征提取，输出序列的时间步长缩减为原来的 1/2。
   * 这等价于将音频特征的采样频率（采样率）降低了一半（例如从 16kHz 降至等效的 8kHz）。
2. **扩大时间域感受野**：
   * 随着时间步长减半，后续层卷积核的一个单位跨度能够覆盖输入信号中两倍的时间跨度，使网络能够捕捉更长时间尺度的语音语调、音节或音乐节奏特征。
3. **信号处理视角（混叠风险）**：
   * 根据奈奎斯特采样定理，如果在下采样之前没有使用合适的低通滤波（在神经网络中由卷积核学习实现），步幅为 2 的操作可能会产生高频混叠（Aliasing）失真。

---

### 习题 4：步幅大于 1 的计算优势是什么？

#### 【解答】
步幅大于 1（`stride > 1`）是深度卷积神经网络中控制模型计算效率的核心手段，主要带来以下四大优势：

1. **大幅削减浮点计算量（FLOPs / MACs）**：
   * 二维特征图在步幅为 `s` 时，输出的高和宽大约缩小至 `1 / s`，输出特征图的空间点数量减少为大约原先的 `1 / (s²)`。
   * 卷积的计算量与输出像素点总数成正比，例如当 `s = 2` 时，该层计算量直接降为接近原来的 **1/4**，在网络浅层输入分辨率极高时提速尤为显著。
2. **显著降低显存占用（Memory Footprint）**：
   * 特征图尺寸的大幅减小不仅减少了前向传播的内存开销，还大幅缩减了反向传播时需要缓存在显存中的中间激活张量（Activation Map）大小。
3. **快速扩大有效感受野（Receptive Field）**：
   * 若步幅始终为 1，网络必须堆叠很多层才能感知全局图像；而步幅大于 1 能呈几何级数快速扩大高层神经元的感受野，以极低的计算成本捕获全局上下文信息。
4. **替代池化层实现可学习的下采样**：
   * 相比于固定规则、不可微调的最大池化（Max Pooling）或平均池化（Average Pooling），步幅大于 1 的带步幅卷积（Strided Convolution）参数是端到端可训练的，网络可以自主学习最优的抗混叠或下采样策略（常用于全卷积网络 All-CNN 或 GAN 生成器/判别器）。